In [ ]:
import sys 
#System(시스템)의 줄임말로, 파이썬 프로그램이 실행되고 있는 컴퓨터 환경 정보를 담고 있는 도구 상자
from pathlib import Path
#Path Library(경로 라이브러리)의 줄임말로, 컴퓨터 속 파일이나 폴더의 위치(경로)를 
# 편리하게 다루기 위해 만들어진 최신 도구 세트

#pathlib이라는 큰 도구 상자 안에 들어있는 핵심 기능(클래스)으로, 컴퓨터 속의 
# '폴더'나 '파일' 그 자체를 추상화한 것

print(sys.executable)
print(Path.cwd())

c:\dev\ai-data-analysis\.venv\Scripts\python.exe
c:\dev\ai-data-analysis\notebooks


STEP 1. 예측 문제와 예측 시점을 먼저 정의합니다


무엇을 예측하는가?
예측 단위는 무엇인가?
언제 예측하는가?
그 시점에 실제로 알 수 있는 정보는 무엇인가?

STEP 2. Feature Leakage Audit  
 머신러닝 모델을 만들 때 가장 치명적인 실수 중 하나인 데이터 누수(Feature Leakage)를 잡아내기 위한 점검표(Audit)

"실전에서 써먹지도 못할 미래 정보나 반칙 데이터를 모델 학습 재료에서 골라내어 버리기 위함"  

📋 점검표 항목 쉽게 이해하기
1. feature: 내가 모델의 학습 재료로 넣으려는 데이터 변수의 이름입니다. (예: 고객의 나이, 과거 구매 횟수, 내일의 날씨 등)
2. 예측 시점 사용 가능: 실제로 모델이 예측을 수행하는 그 순간에 "이 데이터를 진짜로 손에 쥐고 있는가?"를 따지는 것입니다.
	• 예시: 내일 비가 올지 오늘 밤에 예측해야 하는데, '내일 오후의 습도' 데이터를 학습에 쓰면 안 됩니다. 예측 시점에는 알 수 없는 미래 정보이기 때문입니다.
3. Target 계산 재료: 내가 맞추려는 정답(Target)을 만드는 데 직접 사용된 데이터인가?를 확인합니다.
	• 예시: '연간 총 구매 금액'을 맞추는 모델인데, 학습 재료로 '월별 구매 금액'을 넣어버리면 정답을 대놓고 알려주는 꼴이 됩니다.
4. 사후 정보/ID: 사건이 벌어진 이후에나 알 수 있는 정보이거나, 데이터의 순서·고유 번호(ID)처럼 정답의 힌트가 유출되는 항목인지 체크합니다.
5. 최종 사용: 점검 결과 문제가 없다면 O (사용), 반칙 데이터라면 X (제외)를 결정합니다.
6. 판단 이유: 이 데이터를 쓰기로 했거나 버리기로 한 논리적인 근거를 적습니다.



In [4]:
import pandas as pd

# 1. 허용(사용)할 feature와 금지(제외)할 feature 리스트 정의
allowed_features = [
    'order_month', 'order_dayofweek', 'age', 
    'payment_method', 'gender', 'city'
]

forbidden_features = [
    'order_total', 'line_total', 'quantity', 'unit_price', 
    'item_count', 'total_quantity', 'avg_unit_price', 
    'order_status', 'order_id', 'customer_id', 'product_id'
]

# 2. 데이터 파일 직접 불러오기 (성공하신 코드 반영)
df = pd.read_csv("../data/processed/orders_clean.csv")

print("=== 🔍 Feature Leakage Audit 시작 ===")

# ⚠️ [해결 포인트] 지워졌던 변수 정의 부분을 다시 채워 넣었습니다!
current_columns = df.columns.tolist()

# 3. 허용된 데이터 확인
detected_allowed = [col for col in current_columns if col in allowed_features]
print(f"\n✅ 사용 허용된 안전한 Feature ({len(detected_allowed)}개):")
print(detected_allowed)

# 4. 데이터 누수(금지된 데이터) 감지 및 경고
detected_leakage = [col for col in current_columns if col in forbidden_features]

if detected_leakage:
    print(f"\n🚨 [경고] 데이터 누수(Feature Leakage) 위험 컬럼 감지 ({len(detected_leakage)}개)!!")
    print("👉 아래 컬럼들은 모델 학습 전 반드시 제외(drop)해야 합니다:")
    print(detected_leakage)
else:
    print("\n🎉 안전합니다! 금지된 데이터 누수(Feature Leakage) 컬럼이 발견되지 않았습니다.")


=== 🔍 Feature Leakage Audit 시작 ===

✅ 사용 허용된 안전한 Feature (3개):
['payment_method', 'order_month', 'order_dayofweek']

🚨 [경고] 데이터 누수(Feature Leakage) 위험 컬럼 감지 (3개)!!
👉 아래 컬럼들은 모델 학습 전 반드시 제외(drop)해야 합니다:
['order_id', 'customer_id', 'order_status']
